# 04 Llama Structure Blocks Evaluation

## Objective

This notebook evaluates the **LLM structure extraction output** created inside:

```text
data/llama_structure_blocks/
```

Main comparison:

```text
manual reference labels-only blocks JSON
vs
sample*_llama_labels_only_blocks.json
```

This version assumes you already manually created the reference files. It does **not** automatically create reference blocks from text.

Expected manual reference file example:

```text
data/reference_structure_blocks/sample1_reference_labels_only_blocks.json
```

Expected Llama file example:

```text
data/llama_structure_blocks/sample1_llama_labels_only_blocks.json
```


In [1]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
import pandas as pd


## Step 1: Locate Project Root

In [2]:
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

print("Project root:", project_root)
print("Data folder exists:", (project_root / "data").exists())

Project root: c:\Users\Nahid\dmpbridge
Data folder exists: True


## Step 2: Define Input and Output Folders

This notebook uses your **manual gold/reference blocks** from:

```text
data/reference_structure_blocks/
```

It compares them with Llama extracted blocks from:

```text
data/llama_structure_blocks/
```

In [3]:
llama_blocks_dir = project_root / "data" / "llama_structured_blocks"
reference_blocks_dir = project_root / "data" / "reference_structure_blocks"
reports_dir = project_root / "outputs" / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

print("Llama structure blocks folder exists:", llama_blocks_dir.exists())
print("Manual reference blocks folder exists:", reference_blocks_dir.exists())
print("Reports folder:", reports_dir)

Llama structure blocks folder exists: True
Manual reference blocks folder exists: True
Reports folder: c:\Users\Nahid\dmpbridge\outputs\reports


## Step 3: Evaluation Helper Functions

In [4]:
def normalize_label(label: str) -> str:
    """Normalize block labels for fair comparison."""
    label = str(label).strip().lower()
    label = label.replace(" ", "_").replace("-", "_")

    allowed_labels = {"document_title", "section", "subsection", "content"}

    if label in allowed_labels:
        return label

    return "content"


def normalize_eval_text(text: str) -> str:
    """Normalize text for fair word-level evaluation."""
    if not text:
        return ""

    text = unicodedata.normalize("NFKD", text)

    # Fix common PDF ligature artifacts.
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = text.replace("ﬀ", "ff").replace("ﬃ", "ffi").replace("ﬄ", "ffl")

    # Normalize quotes and dashes.
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")

    # Lowercase for matching.
    text = text.lower()

    # Keep words/numbers. Replace punctuation with spaces.
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_words(text: str) -> list[str]:
    """Convert normalized text into word tokens."""
    text = normalize_eval_text(text)
    return text.split() if text else []


def load_blocks_json(path: Path) -> list[dict]:
    """Load a labels-only blocks JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON list in {path}, but got {type(data)}")

    clean_blocks = []

    for block in data:
        if not isinstance(block, dict):
            continue

        label = normalize_label(block.get("label", "content"))
        text = str(block.get("text", "")).strip()

        if not text:
            continue

        clean_blocks.append({
            "label": label,
            "text": text,
        })

    return clean_blocks


def extract_text_from_blocks(blocks: list[dict], include_labels: set[str] | None = None) -> str:
    """Join block text into one comparable text string."""
    texts = []

    for block in blocks:
        label = normalize_label(block.get("label", "content"))
        text = str(block.get("text", "")).strip()

        if not text:
            continue

        if include_labels is not None and label not in include_labels:
            continue

        texts.append(text)

    return "\n".join(texts)


def count_block_labels(blocks: list[dict]) -> dict:
    """Count document_title, section, subsection, and content blocks."""
    counts = Counter(normalize_label(block.get("label", "content")) for block in blocks if isinstance(block, dict))

    return {
        "document_title_count": counts.get("document_title", 0),
        "section_count": counts.get("section", 0),
        "subsection_count": counts.get("subsection", 0),
        "content_count": counts.get("content", 0),
        "total_blocks": sum(counts.values()),
    }


## Step 4: Text Similarity Metrics

In [5]:
def word_counter_overlap(reference_words: list[str], extracted_words: list[str]) -> int:
    """Count overlapping words using frequency-aware matching."""
    ref_counter = Counter(reference_words)
    ext_counter = Counter(extracted_words)
    overlap = ref_counter & ext_counter
    return sum(overlap.values())


def rouge_l_score(reference_words: list[str], extracted_words: list[str]) -> float:
    """Compute a simple ROUGE-L F1 score based on longest common subsequence."""
    if not reference_words or not extracted_words:
        return 0.0

    matcher = SequenceMatcher(None, reference_words, extracted_words)
    lcs = sum(block.size for block in matcher.get_matching_blocks())

    recall = lcs / len(reference_words) if reference_words else 0.0
    precision = lcs / len(extracted_words) if extracted_words else 0.0

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def evaluate_text_pair(reference_text: str, extracted_text: str) -> dict:
    """Evaluate extracted block text against reference block text."""
    reference_words = tokenize_words(reference_text)
    extracted_words = tokenize_words(extracted_text)

    correct_word_count = word_counter_overlap(reference_words, extracted_words)

    reference_word_count = len(reference_words)
    extracted_word_count = len(extracted_words)

    missing_word_count = max(reference_word_count - correct_word_count, 0)
    extra_word_count = max(extracted_word_count - correct_word_count, 0)

    word_recall = correct_word_count / reference_word_count if reference_word_count else 0.0
    word_precision = correct_word_count / extracted_word_count if extracted_word_count else 0.0

    if word_precision + word_recall == 0:
        word_f1 = 0.0
    else:
        word_f1 = 2 * word_precision * word_recall / (word_precision + word_recall)

    return {
        "word_capture": round(word_recall, 4),
        "rouge_l": round(rouge_l_score(reference_words, extracted_words), 4),
        "word_precision": round(word_precision, 4),
        "word_recall": round(word_recall, 4),
        "word_f1": round(word_f1, 4),
        "extracted_word_count": extracted_word_count,
        "reference_word_count": reference_word_count,
        "missing_word_count": missing_word_count,
        "extra_word_count": extra_word_count,
    }


## Step 5: Evaluate One Sample

In [6]:
sample_name = "sample1"

reference_blocks_path = reference_blocks_dir / f"{sample_name}_reference_labels_blocks.json"
llama_blocks_path = llama_blocks_dir / f"{sample_name}_llama_blocks.json"

print("Manual reference blocks exists:", reference_blocks_path.exists())
print("Manual reference path:", reference_blocks_path)
print("Llama blocks exists:", llama_blocks_path.exists())
print("Llama path:", llama_blocks_path)

if not reference_blocks_path.exists():
    raise FileNotFoundError(f"Missing manual reference file: {reference_blocks_path}")

if not llama_blocks_path.exists():
    raise FileNotFoundError(f"Missing Llama blocks file: {llama_blocks_path}")

reference_blocks = load_blocks_json(reference_blocks_path)
llama_blocks = load_blocks_json(llama_blocks_path)

print("Number of manual reference blocks:", len(reference_blocks))
print("Number of Llama blocks:", len(llama_blocks))

Manual reference blocks exists: True
Manual reference path: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks\sample1_reference_labels_blocks.json
Llama blocks exists: True
Llama path: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks\sample1_llama_blocks.json
Number of manual reference blocks: 26
Number of Llama blocks: 23


## Step 6: Preview Manual Reference and Llama Blocks

## Step 7: Evaluate Sample Text and Structure Counts

In [7]:
reference_text_from_blocks = extract_text_from_blocks(reference_blocks)
llama_text_from_blocks = extract_text_from_blocks(llama_blocks)

metrics = evaluate_text_pair(reference_text_from_blocks, llama_text_from_blocks)

reference_counts = count_block_labels(reference_blocks)
llama_counts = count_block_labels(llama_blocks)

metrics.update({
    "sample": sample_name,
    "reference_title_count": reference_counts["document_title_count"],
    "llama_title_count": llama_counts["document_title_count"],
    "reference_section_count": reference_counts["section_count"],
    "llama_section_count": llama_counts["section_count"],
    "reference_subsection_count": reference_counts["subsection_count"],
    "llama_subsection_count": llama_counts["subsection_count"],
    "reference_content_count": reference_counts["content_count"],
    "llama_content_count": llama_counts["content_count"],
    "reference_total_blocks": reference_counts["total_blocks"],
    "llama_total_blocks": llama_counts["total_blocks"],
})

print(json.dumps(metrics, indent=2))

{
  "word_capture": 0.996,
  "rouge_l": 0.997,
  "word_precision": 0.998,
  "word_recall": 0.996,
  "word_f1": 0.997,
  "extracted_word_count": 1003,
  "reference_word_count": 1005,
  "missing_word_count": 4,
  "extra_word_count": 2,
  "sample": "sample1",
  "reference_title_count": 1,
  "llama_title_count": 1,
  "reference_section_count": 6,
  "llama_section_count": 11,
  "reference_subsection_count": 8,
  "llama_subsection_count": 0,
  "reference_content_count": 11,
  "llama_content_count": 11,
  "reference_total_blocks": 26,
  "llama_total_blocks": 23
}


## Step 8: Block-by-Block Comparison

In [8]:
comparison_rows = []
max_len = max(len(reference_blocks), len(llama_blocks))

for i in range(max_len):
    ref_block = reference_blocks[i] if i < len(reference_blocks) else {}
    llama_block = llama_blocks[i] if i < len(llama_blocks) else {}

    reference_label = ref_block.get("label", "")
    llama_label = llama_block.get("label", "")
    reference_text = ref_block.get("text", "")
    llama_text = llama_block.get("text", "")

    comparison_rows.append({
        "block_index": i + 1,
        "reference_label": reference_label,
        "reference_text": reference_text,
        "llama_label": llama_label,
        "llama_text": llama_text,
        "label_match": reference_label == llama_label,
        "text_exact_match": normalize_eval_text(reference_text) == normalize_eval_text(llama_text),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head(30)

,block_index,reference_label,reference_text,llama_label,llama_text,label_match,text_exact_match
0,1,document_title,DATA MANAGEMENT AND SHARING PLAN,document_title,DATA MANAGEMENT AND SHARING PLAN,True,True
1,2,section,Element 1: Data Type:,section,Element 1: Data Type:,True,True
2,3,subsection,A. Types and amount of scientific data expecte...,content,A. Types and amount of scientific data expecte...,False,False
3,4,content,This secondary data analysis project will anal...,section,B. Scientific data that will be preserved and ...,False,False
4,5,subsection,B. Scientific data that will be preserved and ...,content,"As this is a secondary data analysis project, ...",False,False
5,6,content,"As this is a secondary data analysis project, ...",section,"C. Metadata, other relevant data, and associat...",False,False
6,7,subsection,"C. Metadata, other relevant data, and associat...",content,"In addition to the data described above, code ...",False,False
7,8,content,"In addition to the data described above, code ...",section,"Element 2: Related Tools, Software and/or Code:",False,False
8,9,section,"Element 2: Related Tools, Software and/or Code:",content,Data will be analyzed with custom code by our ...,False,False
9,10,content,Data will be analyzed with custom code by our ...,section,Element 3: Standards:,False,False


## Step 9: Save Sample Comparison

In [9]:
sample_comparison_path = reports_dir / f"{sample_name}_llama_blocks_vs_manual_reference_blocks_comparison.csv"
comparison_df.to_csv(sample_comparison_path, index=False)

print("Saved comparison CSV:", sample_comparison_path)

Saved comparison CSV: c:\Users\Nahid\dmpbridge\outputs\reports\sample1_llama_blocks_vs_manual_reference_blocks_comparison.csv


## Step 10: Batch Evaluation for All Samples

In [10]:
results = []

for llama_blocks_path in sorted(llama_blocks_dir.glob("*_llama_labels_only_blocks.json")):
    sample_name = llama_blocks_path.name.replace("_llama_labels_only_blocks.json", "")
    reference_blocks_path = reference_blocks_dir / f"{sample_name}_reference_labels_only_blocks.json"

    if not reference_blocks_path.exists():
        print(f"Skipping {sample_name}: missing manual reference blocks JSON")
        continue

    llama_blocks = load_blocks_json(llama_blocks_path)
    reference_blocks = load_blocks_json(reference_blocks_path)

    reference_text_from_blocks = extract_text_from_blocks(reference_blocks)
    llama_text_from_blocks = extract_text_from_blocks(llama_blocks)

    metrics = evaluate_text_pair(reference_text_from_blocks, llama_text_from_blocks)

    reference_counts = count_block_labels(reference_blocks)
    llama_counts = count_block_labels(llama_blocks)

    metrics.update({
        "sample": sample_name,
        "reference_title_count": reference_counts["document_title_count"],
        "llama_title_count": llama_counts["document_title_count"],
        "reference_section_count": reference_counts["section_count"],
        "llama_section_count": llama_counts["section_count"],
        "reference_subsection_count": reference_counts["subsection_count"],
        "llama_subsection_count": llama_counts["subsection_count"],
        "reference_content_count": reference_counts["content_count"],
        "llama_content_count": llama_counts["content_count"],
        "reference_total_blocks": reference_counts["total_blocks"],
        "llama_total_blocks": llama_counts["total_blocks"],
    })

    results.append(metrics)

results_df = pd.DataFrame(results)
results_df

""


## Step 11: Save Batch Evaluation Results

In [11]:
evaluation_output_path = reports_dir / "llama_structure_blocks_vs_manual_reference_blocks_evaluation.csv"

if not results_df.empty:
    results_df.to_csv(evaluation_output_path, index=False)
    print("Saved evaluation results:", evaluation_output_path)
else:
    print("No results found. Check whether manual reference files exist in:", reference_blocks_dir)

No results found. Check whether manual reference files exist in: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks


## Step 12: Summary Statistics

In [12]:
if not results_df.empty:
    metric_columns = [
        "word_capture",
        "rouge_l",
        "word_precision",
        "word_recall",
        "word_f1",
        "extracted_word_count",
        "reference_word_count",
        "missing_word_count",
        "extra_word_count",
        "reference_section_count",
        "llama_section_count",
        "reference_subsection_count",
        "llama_subsection_count",
    ]

    summary_df = results_df[metric_columns].agg(["mean", "std", "min", "max"]).round(4)
    display(summary_df)
else:
    print("No results to summarize.")

No results to summarize.


## Interpretation Notes

- `word_recall` / `word_capture`: how much manual reference content Llama preserved.
- `word_precision`: how much of the Llama output is correct compared with the manual reference.
- `word_f1`: balance between precision and recall.
- `rouge_l`: whether the word order is close to the manual reference.
- title/section/subsection counts: whether Llama detected the same structure as your manual gold reference.

This notebook does **not** generate reference files. Your manual reference JSON files are treated as the gold standard.
